In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.data import load_data

df = load_data()
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_rate,personal_status,other_parties,...,property,age,other_payment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [2]:
df["monthly_payment"] = df["credit_amount"] / df["duration"]

df[['monthly_payment', 'class']].groupby('class').describe()

monthly_payment                                                \
                count        mean         std        min        25%   
class                                                                 
1               700.0  165.819730  110.574834  25.250000  96.906250   
2               300.0  172.044031  223.840381  24.055556  76.197917   

                                            
              50%         75%          max  
class                                       
1      134.916667  211.975000  1126.833333  
2      121.058333  196.361111  2482.666667

In [3]:
checking_map = {  
  'A11': 0,  
  'A12': 1,  
  'A13': 2, 
  'A14': 3 
}

# Map savings status to points
savings_map = {
  'A61': 0, 'A62': 1, 
  'A63': 2, 'A64': 3, 'A65': 2   
}

df['checking_points'] = df['checking_status'].map(checking_map)
df['savings_points'] = df['savings_status'].map(savings_map)

df['financial_stability'] = df['checking_points'] + df['savings_points']

# Check correlation with target
pd.crosstab(df['financial_stability'], df['class'], normalize='index')

class,1,2
financial_stability,,
0,0.479452,0.520548
1,0.560976,0.439024
2,0.600000,0.400000
3,0.856589,0.143411
4,0.835821,0.164179
5,0.901408,0.098592
6,0.920000,0.080000


In [4]:
employment_map = {
  'A71': 0, 'A72': 0,
  'A73': 1,
  'A74': 2, 'A75': 2
}
# employment mapping
df['employment_points'] = df['employment'].map(employment_map)
print(pd.crosstab(df['employment_points'], df['class'], normalize='index'))

#overall mapping
df['overall_stability'] = df['financial_stability'] + df['employment_points']
print(pd.crosstab(df['overall_stability'], df['class'], normalize='index'))

class                     1         2
employment_points                    
0                  0.602564  0.397436
1                  0.693215  0.306785
2                  0.758782  0.241218
class                     1         2
overall_stability                    
0                  0.440678  0.559322
1                  0.519685  0.480315
2                  0.514793  0.485207
3                  0.631250  0.368750
4                  0.800000  0.200000
5                  0.889610  0.110390
6                  0.835616  0.164384
7                  0.954545  0.045455
8                  0.933333  0.066667


In [5]:
purpose_map = {
  **dict.fromkeys(['A40', 'A410', 'A46'], 0),
  **dict.fromkeys(['A42', 'A44', 'A45', 'A49'], 1),
  **dict.fromkeys(['A41', 'A43'], 2),
  'A48': 3
}

df['purpose_points'] = df['purpose'].map(purpose_map)
print(pd.crosstab(df['purpose_points'], df['class'], normalize='index'))

class                  1         2
purpose_points                    
0               0.608108  0.391892
1               0.666667  0.333333
2               0.793734  0.206266
3               0.888889  0.111111


In [6]:
feature_columns = [
    'duration', 'credit_amount', 'monthly_payment',
    'age', 'installment_rate', 'residence_since', 'existing_credits', 'num_dependents',
    'financial_stability', 'employment_points', 'overall_stability', 'purpose_points',
    'credit_history', 'personal_status', 'other_parties', 'property', 
    'other_payment_plans', 'housing', 'job', 'telephone', 'foreign_worker',
    'class'
]

df_engineered = df[feature_columns]
df_engineered.to_csv('../data/processed/german_credit_engineered.csv', index=False)
print("Engineered features saved!")

Engineered features saved!


## Key Findings:

1. **overall_stability** (financial + employment): 50% default spread — strongest predictor
2. **financial_stability** (checking + savings): 44% spread — second strongest
3. **purpose_points**: 28% spread — useful grouping
4. **employment_points**: 16% spread — marginal improvement
5. **monthly_payment**: Weak and counterintuitive (bad borrowers have lower monthly payments due to longer durations)

Next: Encode categorical features and train baseline models

In [7]:
processed_df = pd.read_csv('../data/processed/german_credit_engineered.csv')

processed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   duration             1000 non-null   int64  
 1   credit_amount        1000 non-null   int64  
 2   monthly_payment      1000 non-null   float64
 3   age                  1000 non-null   int64  
 4   installment_rate     1000 non-null   int64  
 5   residence_since      1000 non-null   int64  
 6   existing_credits     1000 non-null   int64  
 7   num_dependents       1000 non-null   int64  
 8   financial_stability  1000 non-null   int64  
 9   employment_points    1000 non-null   int64  
 10  overall_stability    1000 non-null   int64  
 11  purpose_points       1000 non-null   int64  
 12  credit_history       1000 non-null   object 
 13  personal_status      1000 non-null   object 
 14  other_parties        1000 non-null   object 
 15  property             1000 non-null   ob

In [8]:
df_encoded = pd.get_dummies(processed_df,	drop_first=True)

df_encoded['class'] = df_encoded['class'].map({1: 0, 2: 1})

print(df_encoded.head(10))
df_encoded.info()

   duration  credit_amount  monthly_payment  age  installment_rate  \
0         6           1169       194.833333   67                 4   
1        48           5951       123.979167   22                 2   
2        12           2096       174.666667   49                 2   
3        42           7882       187.666667   45                 2   
4        24           4870       202.916667   53                 3   
5        36           9055       251.527778   35                 2   
6        24           2835       118.125000   53                 3   
7        36           6948       193.000000   35                 2   
8        12           3059       254.916667   61                 2   
9        30           5234       174.466667   28                 4   

   residence_since  existing_credits  num_dependents  financial_stability  \
0                4                 2               1                    2   
1                2                 1               1                    1  

In [9]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('class', axis=1)
y = df_encoded['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")

Train: 800, Test: 200
Train class distribution:
class
0    0.69875
1    0.30125
Name: proportion, dtype: float64


In [10]:
#  dropping the idea of SMOTE we will only depend on the good metrics value likely AUC-ROC curve, F1-Score.

# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42)
# X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# print(f"Before SMOTE: {y_train.value_counts()}")
# print(f"After SMOTE: {y_train_balanced.value_counts()}")

In [11]:
import joblib
joblib.dump((X_train, X_test, y_train, y_test), '../data/processed/train_test_split.pkl')
print("Train-test split saved!")

Train-test split saved!
